# WANDB : Weights and Biases
### The standard way Weights & Biases searches for the best settings

<img src="http://wandb.me/logo-im-png" width="300" alt="Weights & Biases" />

In notebook **06b** we ran nine experiments **by hand** — we wrote out each setting and looped over them. That works, but it is slow and we can only test what we think of.

A **Sweep** is the standard W&B tool for this. You describe the settings to try, and W&B **runs the search for you** and finds the best combination. Same model (TinyVGG), same data (FashionMNIST) — we just let W&B do the searching.

**A sweep has three parts:**

1. **The sweep config** — *what* to try (the settings and their ranges) and *what* to optimise (e.g. highest test accuracy).
2. **A training function** — trains **one** model using the settings W&B hands it.
3. **An agent** — asks W&B for settings, runs the training function, repeats.

The two key calls are `wandb.sweep(...)` (create the search) and `wandb.agent(...)` (run it).

---
## 0. Setup

Install, log in, and set the device, seed, and project name.

In [ ]:
!pip install wandb torchinfo -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 40.4 MB/s eta 0:00:00


In [ ]:
import wandb
wandb.login()

True

In [ ]:
import os
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seeds(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seeds()

PROJECT = "tinyvgg-fashionmnist-sweep"
print("device:", device, "| project:", PROJECT)

device: cuda | project: tinyvgg-fashionmnist-sweep


---
## 1. Prerequisites (same building blocks as before)

These are the exact pieces from the previous notebooks: download the data, build the transforms and dataloaders, define **TinyVGG**, and the `train_step` / `test_step` functions. We need them so the sweep's training function has something to train. They are unchanged — read quickly and move on.

In [ ]:
import zipfile
from pathlib import Path
import requests

def download_data(source: str, destination: str, remove_source: bool = True) -> Path:
  """Downloads a zipped dataset from source and unzips to destination."""
  data_path = Path("data/")
  image_path = data_path / destination
  if image_path.is_dir():
    print(f"[INFO] {image_path} already exists, skipping download.")
  else:
    image_path.mkdir(parents=True, exist_ok=True)
    target_file = Path(source).name
    with open(data_path / target_file, "wb") as f:
      print(f"[INFO] Downloading {target_file}...")
      f.write(requests.get(source).content)
    with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
      print(f"[INFO] Unzipping {target_file}...")
      zip_ref.extractall(image_path)
    if remove_source:
      os.remove(data_path / target_file)
  return image_path

data_10_percent_path = download_data(
    "https://github.com/alpha023/dataset/raw/main/fashion_mnist_10pct.zip", "fashion_mnist_10pct")
data_20_percent_path = download_data(
    "https://github.com/alpha023/dataset/raw/main/fashion_mnist_20pct.zip", "fashion_mnist_20pct")

train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"
test_dir = data_10_percent_path / "test"

[INFO] Downloading fashion_mnist_10pct.zip...
[INFO] Unzipping fashion_mnist_10pct.zip...
[INFO] Downloading fashion_mnist_20pct.zip...
[INFO] Unzipping fashion_mnist_20pct.zip...


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

# Transform: resize -> tensor -> normalize
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
simple_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    normalize,
])

def create_dataloaders(train_dir, test_dir, transform, batch_size, num_workers=NUM_WORKERS):
  """Turns image folders into train and test DataLoaders."""
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)
  class_names = train_data.classes
  train_dl = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                        num_workers=num_workers, pin_memory=True)
  test_dl = DataLoader(test_data, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True)
  return train_dl, test_dl, class_names

In [ ]:
class TinyVGG(nn.Module):
    """TinyVGG from https://poloclub.github.io/cnn-explainer/"""
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(input_shape, hidden_units, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, 3, 1, 1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 16 * 16, output_shape),
        )

    def forward(self, x):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

In [ ]:
from typing import Tuple

def train_step(model, dataloader, loss_fn, optimizer, device) -> Tuple[float, float]:
    model.train()
    train_loss, train_acc = 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)
    return train_loss / len(dataloader), train_acc / len(dataloader)

def test_step(model, dataloader, loss_fn, device) -> Tuple[float, float]:
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            test_loss += loss_fn(logits, y).item()
            test_acc += (logits.argmax(dim=1) == y).sum().item() / len(logits)
    return test_loss / len(dataloader), test_acc / len(dataloader)

---
## 2. Step 1 of a sweep — the sweep config

The sweep config is a Python dictionary with three keys:

**`method`** — *how* to search. There are three standard choices:

| Method | What it does | When to use |
|---|---|---|
| `grid` | tries **every** combination | small, discrete options; you want to be thorough |
| `random` | picks **random** combinations | a quick, cheap search; works with continuous ranges |
| `bayes` | **learns** from past runs to pick promising settings next | expensive runs; you want efficiency (needs a metric) |

**`metric`** — the number to optimise. We give its `name` (it must match a value we `wandb.log`) and a `goal` (`maximize` or `minimize`).

**`parameters`** — the settings to try. Each one can be:
- `{"value": x}` — a **fixed** value (kept constant),
- `{"values": [a, b, c]}` — a **list** of choices,
- `{"min": a, "max": b}` — a **range** (uniform),
- `{"distribution": "log_uniform_values", "min": a, "max": b}` — a range sampled on a **log scale** (the standard choice for learning rate).

Below we use **Bayesian** search, optimise **test accuracy**, and search over hidden units, batch size, optimizer, and learning rate.

In [ ]:
sweep_config = {
    "method": "bayes",                                    # smart search
    "metric": {"name": "test/acc", "goal": "maximize"},   # optimise test accuracy
    "parameters": {
        "dataset":      {"values": ["20pct","10pct"]},                                       # fixed
        "epochs":       {"values": [1,2,3,4]},                                             # fixed
        "hidden_units": {"values": [16, 32, 64]},                                 # choices
        "batch_size":   {"values": [16, 32, 64]},                                 # choices
        "optimizer":    {"values": ["adam", "sgd", "rmsprop"]},                   # choices
        "lr":           {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e-2},  # range (log scale)
    },
}

sweep_config

{'method': 'bayes',
 'metric': {'name': 'test/acc', 'goal': 'maximize'},
 'parameters': {'dataset': {'value': ['20pct', '10pct']},
  'epochs': {'value': 3},
  'hidden_units': {'values': [16, 32, 64]},
  'batch_size': {'values': [16, 32, 64]},
  'optimizer': {'values': ['adam', 'sgd', 'rmsprop']},
  'lr': {'distribution': 'log_uniform_values', 'min': 0.0001, 'max': 0.01}}}

---
## 3. Step 2 of a sweep — the training function

This function trains **one** model. The agent will call it many times, each time with a **different** config chosen by the sweep.

Key points (this is the standard W&B pattern):

- We start with `wandb.init()` and **do not pass a config** — the sweep fills it in. We then read the settings from `wandb.config`.
- We use `with wandb.init() as run:` so the run closes automatically at the end.
- We **log the metric the sweep optimises** (`test/acc`) with `wandb.log`. The name must match the `metric` in the config.
- `wandb.define_metric("test/acc", summary="max")` tells W&B to use the **best** epoch's accuracy as the run's score (not just the last epoch).

In [ ]:
def train_sweep(config=None):
    # The agent passes the config in; wandb.init picks it up.
    with wandb.init(config=config):
        config = wandb.config
        set_seeds()

        # Use the best (max) accuracy and best (min) loss as the run's score
        wandb.define_metric("test/acc", summary="max")
        wandb.define_metric("test/loss", summary="min")

        # 1) Data
        train_dir = train_dir_10_percent if config.dataset == "10pct" else train_dir_20_percent
        train_dl, test_dl, class_names = create_dataloaders(
            train_dir, test_dir, simple_transform, config.batch_size)

        # 2) Model
        model = TinyVGG(input_shape=3, hidden_units=config.hidden_units,
                        output_shape=len(class_names)).to(device)
        loss_fn = nn.CrossEntropyLoss()

        # 3) Optimizer (chosen by the sweep)
        if config.optimizer == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
        elif config.optimizer == "sgd":
            optimizer = torch.optim.SGD(model.parameters(), lr=config.lr, momentum=0.9)
        else:
            optimizer = torch.optim.RMSprop(model.parameters(), lr=config.lr)

        # 4) Train, logging the metric the sweep optimises
        for epoch in range(config.epochs):
            train_loss, train_acc = train_step(model, train_dl, loss_fn, optimizer, device)
            test_loss, test_acc = test_step(model, test_dl, loss_fn, device)
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss, "train/acc": train_acc,
                "test/loss": test_loss,   "test/acc": test_acc,
            })
            print(f"  epoch {epoch+1}: test_acc={test_acc:.4f}")

---
## 4. Step 3a — create the sweep

`wandb.sweep` registers the search on W&B and returns a **sweep id**. Nothing trains yet — this just sets up the search.

In [ ]:
sweep_id = wandb.sweep(sweep_config, project=PROJECT)
print("Sweep id:", sweep_id)

Create sweep with ID: hc6rwla0
Sweep URL: https://wandb.ai/p25ai0001-indian-institute-of-technology-jodhpur/tinyvgg-fashionmnist-sweep/sweeps/hc6rwla0
Sweep id: hc6rwla0


---
## 5. Step 3b — run the sweep with an agent

`wandb.agent` starts the search. For each trial it:
1. asks W&B for a set of settings,
2. calls our `train_sweep` function with them,
3. sends the results back.

`count=15` means "run 15 trials". With Bayesian search, later trials use what earlier trials learned, so accuracy tends to improve as it goes.

> Tip: you can run this **same cell on another machine or Colab** with the same `sweep_id` to add more agents and search in **parallel**.

In [ ]:
wandb.agent(sweep_id, function=train_sweep, count=15)
print("Sweep finished. Open the sweep link above to see the results.")

wandb: Agent Starting Run: rgbrvow8 with config:
wandb: 	batch_size: 64
wandb: 	dataset: ['20pct', '10pct']
wandb: 	epochs: 3
wandb: 	hidden_units: 64
wandb: 	lr: 0.0028855626199432503
wandb: 	optimizer: rmsprop
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  epoch 1: test_acc=0.7906
  epoch 2: test_acc=0.7799
  epoch 3: test_acc=0.8293


epoch,▁▅█
test/acc,▃▁█
test/loss,▆█▁
train/acc,▁▆█
train/loss,█▁▁
epoch,3
train/acc,0.81034
train/loss,0.53695


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 5jguku32 with config:
wandb: 	batch_size: 16
wandb: 	dataset: ['20pct', '10pct']
wandb: 	epochs: 3
wandb: 	hidden_units: 64
wandb: 	lr: 0.001126546033981276
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  epoch 1: test_acc=0.8839
  epoch 2: test_acc=0.8919
  epoch 3: test_acc=0.9008


...

epoch,▁▅█
test/acc,▁▅█
test/loss,█▃▁
train/acc,▁▆█
train/loss,█▂▁
epoch,3
train/acc,0.88015
train/loss,0.34678


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 6yff30wr with config:
wandb: 	batch_size: 16
wandb: 	dataset: ['20pct', '10pct']
wandb: 	epochs: 3
wandb: 	hidden_units: 16
wandb: 	lr: 0.0001183668748110101
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  epoch 1: test_acc=0.8234
  epoch 2: test_acc=0.8562
  epoch 3: test_acc=0.8720


epoch,▁▅█
test/acc,▁▆█
test/loss,█▃▁
train/acc,▁▇█
train/loss,█▂▁
epoch,3
train/acc,0.86067
train/loss,0.3917


Sweep finished. Open the sweep link above to see the results.


---
## 6. Read the results

Open the **sweep page** (link printed above). W&B gives you two charts that are made for tuning:

- **Parallel coordinates** — each line is one run; follow the lines that reach the highest accuracy to see which settings they used.
- **Parameter importance** — a bar chart showing which settings mattered most for accuracy (and in which direction).

You can also get the best run **in code**, using the W&B API.

In [ ]:
api = wandb.Api()
entity = api.default_entity                       # your username/team
sweep = api.sweep(f"{entity}/{PROJECT}/{sweep_id}")

best_run = sweep.best_run()                        # best by the sweep's metric
print("Best run name:", best_run.name)
print("Best test/acc:", best_run.summary.get("test/acc"))
print("Best settings:", {k: v for k, v in best_run.config.items() if not k.startswith("_")})

wandb: Sorting runs by -summary_metrics.test/acc


Best run name: major-sweep-14
Best test/acc: {'max': 0.8771484375}
Best settings: {'lr': 0.00021399074000963823, 'epochs': 3, 'dataset': ['20pct', '10pct'], 'optimizer': 'rmsprop', 'batch_size': 64, 'hidden_units': 32}


---
## 7. Get a usable model from the best settings

A plain sweep tracks **scores**, not saved models. To get a trained model from the winning settings, the simplest way is to **train once more** with the best config.

(In production you would instead log a `wandb.Artifact` inside `train_sweep` so every run saves its model — but retraining the single best one is cleaner for a demo.)

In [ ]:
best_cfg = {k: v for k, v in best_run.config.items() if not k.startswith("_")}
print("Training final model with:", best_cfg)

set_seeds()
train_dir = train_dir_10_percent if best_cfg["dataset"] == "10pct" else train_dir_20_percent
train_dl, test_dl, class_names = create_dataloaders(
    train_dir, test_dir, simple_transform, best_cfg["batch_size"])

best_model = TinyVGG(input_shape=3, hidden_units=best_cfg["hidden_units"],
                     output_shape=len(class_names)).to(device)
loss_fn = nn.CrossEntropyLoss()

if best_cfg["optimizer"] == "adam":
    optimizer = torch.optim.Adam(best_model.parameters(), lr=best_cfg["lr"])
elif best_cfg["optimizer"] == "sgd":
    optimizer = torch.optim.SGD(best_model.parameters(), lr=best_cfg["lr"], momentum=0.9)
else:
    optimizer = torch.optim.RMSprop(best_model.parameters(), lr=best_cfg["lr"])

for epoch in range(best_cfg["epochs"]):
    tr_loss, tr_acc = train_step(best_model, train_dl, loss_fn, optimizer, device)
    te_loss, te_acc = test_step(best_model, test_dl, loss_fn, device)
    print(f"epoch {epoch+1}: test_acc={te_acc:.4f}")

# Save it
Path("models").mkdir(exist_ok=True)
torch.save(best_model.state_dict(), "models/best_sweep_model.pth")
print("Saved best model to models/best_sweep_model.pth")

Training final model with: {'lr': 0.00021399074000963823, 'epochs': 3, 'dataset': ['20pct', '10pct'], 'optimizer': 'rmsprop', 'batch_size': 64, 'hidden_units': 32}
epoch 1: test_acc=0.8273
epoch 2: test_acc=0.8635
epoch 3: test_acc=0.8781
Saved best model to models/best_sweep_model.pth


---
## 8. (Optional) Stop bad runs early with Hyperband

When you run many trials, some are clearly losing after just one or two epochs. **Early termination** stops those early and spends the time on promising runs instead. The standard method in W&B is **Hyperband**.

You add one key to the config and create a new sweep:

```python
sweep_config["early_terminate"] = {
    "type": "hyperband",
    "min_iter": 2,    # let every run reach at least 2 logged steps
    "eta": 2,         # at each checkpoint, keep the better half
}
```

This needs the metric to be logged each epoch (we already do that). Below is a ready-to-run version — uncomment to try it.

In [ ]:
# sweep_config_es = dict(sweep_config)
# sweep_config_es["early_terminate"] = {"type": "hyperband", "min_iter": 2, "eta": 2}
#
# sweep_id_es = wandb.sweep(sweep_config_es, project=PROJECT)
# wandb.agent(sweep_id_es, function=train_sweep, count=15)
print("Early-termination example is shown above (left commented).")

Early-termination example is shown above (left commented).


---
## 9. Recap

You have now done hyperparameter tuning the standard W&B way:

1. **Sweep config** — chose a search `method` (grid / random / **bayes**), a `metric` to optimise, and the `parameters` to try.
2. **Training function** — read settings from `wandb.config` and logged the metric.
3. **`wandb.sweep`** to create the search, **`wandb.agent`** to run it.
4. Read the **parallel coordinates** and **parameter importance** charts, and pulled the **best run** with the W&B API.
5. (Optional) **Hyperband** early termination to save time.

This replaces the manual experiment loop from notebook 06b: instead of you writing out each trial, W&B plans and runs the search, and tells you the winning settings.